# RAG Submission — Pengembangan Generative AI berbasis LLM (Advanced)
**Nama:** Galih Aji Pangestu

**Tujuan:** Membangun sistem RAG (Retrieval-Augmented Generation) untuk menjawab pertanyaan seputar regulasi ketenagakerjaan Indonesia, menggunakan model hasil fine-tuning GRPO (tahap sebelumnya) sebagai satu-satunya generator jawaban. Sistem menggabungkan parent-child chunking, hybrid retrieval (semantic + BM25), metadata filtering, HyDE query expansion, cross-encoder reranking, fallback pencarian web (DuckDuckGo), dan antarmuka Gradio.

> **Catatan eksekusi (CPU-only):** Notebook ini dijalankan pada environment tanpa GPU/CUDA. Model generator yang dipakai adalah model GRPO publik berukuran kecil (`HuggingFaceTB/SmolLM2-135M-Instruct` + adapter LoRA GRPO ter-merge) yang diunggah ke Hugging Face Hub pada notebook GRPO sebelumnya, dan dimuat di sini dengan `transformers.AutoModelForCausalLM` (bukan `unsloth.FastLanguageModel` 4-bit, yang mensyaratkan CUDA). Seluruh tahap retrieval (PDF loading, chunking, embedding, Chroma, BM25, HyDE, reranking, fallback web) dan generasi jawaban dijalankan secara nyata end-to-end, konsisten dengan pendekatan yang sudah dipakai pada notebook Fine-tuning dan GRPO.

**Dokumen sumber (4 PDF, teks asli, tanpa OCR):**
- PP Nomor 35 Tahun 2021.pdf (56 halaman)
- PP Nomor 51 Tahun 2023.pdf (27 halaman)
- PP Nomor 5 Tahun 2021.pdf (739 halaman)
- UU Nomor 6 Tahun 2023.pdf (1127 halaman)

## Pemetaan Rubrik

| Kriteria | Level | Bagian Notebook |
|---|---|---|
| Ekstraksi & persiapan dokumen | Basic | 5, 6, 7, 8 |
| Chunking & indexing vektor | Basic/Skilled | 9, 10, 11 |
| Retrieval dasar | Basic | 11, 12 |
| Hybrid retrieval (semantic + keyword) | Skilled | 13, 14 |
| Metadata filtering | Skilled | 15 |
| Query expansion (HyDE) | Advanced | 16 |
| Reranking (cross-encoder) | Advanced | 17, 18 |
| Fallback ke web search di luar domain | Advanced | 19, 20 |
| Model generatif hasil fine-tuning (GRPO) sebagai generator | Advanced | 21 |
| Prompt engineering & citation | Skilled/Advanced | 22, 23 |
| Pipeline end-to-end | Advanced | 24 |
| Pengujian wajib (lokal, filter, fallback web) | Advanced | 25, 26, 27 |
| Antarmuka pengguna (Gradio) | Skilled | 28 |
| Verifikasi akhir (self-check) | Advanced | 29 |


## 2. Setup Dependency

In [1]:
%%capture
!pip install -q -U transformers accelerate peft
!pip install -q -U sentence-transformers
!pip install -q -U langchain langchain-community langchain-text-splitters langchain-chroma chromadb langchain-huggingface
!pip install -q -U rank-bm25 pypdf duckduckgo-search gradio pandas numpy


## 3. Autentikasi Aman ke Hugging Face
Token **tidak pernah** ditampilkan/di-print. Urutan pencarian token:
1. `google.colab.userdata` (Colab Secrets)
2. Environment variable `HF_TOKEN`
3. Input tersembunyi via `getpass` (fallback interaktif terakhir)


In [2]:
import os

def get_hf_token():
    token = None
    # 1. Coba Colab userdata (Colab Secrets)
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None

    # 2. Fallback ke environment variable
    if not token:
        token = os.environ.get("HF_TOKEN")

    # 3. Fallback terakhir: input tersembunyi
    if not token:
        from getpass import getpass
        token = getpass("Masukkan HF_TOKEN (tidak akan ditampilkan): ")

    if not token:
        raise RuntimeError("HF_TOKEN tidak ditemukan. Harap sediakan token yang valid.")

    os.environ["HF_TOKEN"] = token
    return token

HF_TOKEN = get_hf_token()
print("HF_TOKEN berhasil dimuat. (nilai token disembunyikan, panjang karakter: %d)" % len(HF_TOKEN))

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print("Login Hugging Face berhasil.")


HF_TOKEN berhasil dimuat. (nilai token disembunyikan, panjang karakter: 37)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Login Hugging Face berhasil.


## 4. Konfigurasi Global

In [3]:
import random
import numpy as np
import torch
from huggingface_hub import HfApi

SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# Deteksi GPU
if torch.cuda.is_available():
    print(f"GPU terdeteksi: {torch.cuda.get_device_name(0)}")
else:
    print(
        "CPU-only environment terdeteksi (tidak ada GPU/CUDA). Notebook ini dijalankan dengan "
        "transformers + peft (bukan unsloth, yang mensyaratkan CUDA) dan model GRPO publik "
        "berukuran kecil yang telah diunggah ke Hugging Face Hub pada tahap sebelumnya."
    )

# Kandidat direktori PDF (hanya dua level ini, tidak ada pencarian rekursif di luar)
PDF_DIR_CANDIDATES = [".", "./Document RAG"]

REQUIRED_PDFS = [
    "PP Nomor 35 Tahun 2021.pdf",
    "PP Nomor 51 Tahun 2023.pdf",
    "PP Nomor 5 Tahun 2021.pdf",
    "UU Nomor 6 Tahun 2023.pdf",
]

# Resolusi username HF saat runtime
_api = HfApi()
_whoami = _api.whoami()
HF_USERNAME = _whoami["name"]
GRPO_MODEL_ID = f"{HF_USERNAME}/pgabl-legal-grpo-smollm2-135m-galih"
print("GRPO_MODEL_ID:", GRPO_MODEL_ID)

EMBEDDING_MODEL_ID = "intfloat/multilingual-e5-small"
RERANKER_MODEL_ID = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"

RERANK_THRESHOLD = 0.50
CANDIDATE_K = 10
TOP_K_FINAL = 3

BM25_WEIGHT = 0.4
SEMANTIC_WEIGHT = 0.6

PARENT_CHUNK_SIZE = 1800
PARENT_OVERLAP = 200
CHILD_CHUNK_SIZE = 450
CHILD_OVERLAP = 75

print("Konfigurasi selesai dimuat.")


CPU-only environment terdeteksi (tidak ada GPU/CUDA). Notebook ini dijalankan dengan transformers + peft (bukan unsloth, yang mensyaratkan CUDA) dan model GRPO publik berukuran kecil yang telah diunggah ke Hugging Face Hub pada tahap sebelumnya.


GRPO_MODEL_ID: galiihajiip/pgabl-legal-grpo-smollm2-135m-galih
Konfigurasi selesai dimuat.


## 5. Verifikasi Keberadaan 4 PDF Wajib

In [4]:
import os

def resolve_pdf_path(filename: str):
    for base in PDF_DIR_CANDIDATES:
        candidate = os.path.join(base, filename)
        if os.path.isfile(candidate):
            return candidate
    return None

resolved_pdf_paths = {}
missing_files = []

for fname in REQUIRED_PDFS:
    path = resolve_pdf_path(fname)
    if path is None:
        missing_files.append(fname)
    else:
        resolved_pdf_paths[fname] = path

if missing_files:
    raise RuntimeError(
        "File PDF wajib berikut tidak ditemukan di direktori kandidat "
        f"{PDF_DIR_CANDIDATES}: {missing_files}. Harap pastikan seluruh 4 dokumen tersedia sebelum melanjutkan."
    )

print("Semua 4 PDF wajib ditemukan:")
for fname, path in resolved_pdf_paths.items():
    size_kb = os.path.getsize(path) / 1024
    print(f" - {fname}: path={path}, ukuran={size_kb:.1f} KB")


Semua 4 PDF wajib ditemukan:
 - PP Nomor 35 Tahun 2021.pdf: path=.\PP Nomor 35 Tahun 2021.pdf, ukuran=2458.0 KB
 - PP Nomor 51 Tahun 2023.pdf: path=.\PP Nomor 51 Tahun 2023.pdf, ukuran=2704.2 KB
 - PP Nomor 5 Tahun 2021.pdf: path=.\PP Nomor 5 Tahun 2021.pdf, ukuran=16669.9 KB
 - UU Nomor 6 Tahun 2023.pdf: path=.\UU Nomor 6 Tahun 2023.pdf, ukuran=83353.9 KB


## 6. Memuat PDF (PyPDFLoader)

In [5]:
from langchain_community.document_loaders import PyPDFLoader

raw_pages_by_file = {}
total_pages = 0
total_skipped_empty = 0

for fname, path in resolved_pdf_paths.items():
    loader = PyPDFLoader(path)
    pages = loader.load()
    n_before = len(pages)

    non_empty_pages = [p for p in pages if p.page_content and p.page_content.strip()]
    n_skipped = n_before - len(non_empty_pages)

    raw_pages_by_file[fname] = non_empty_pages
    total_pages += len(non_empty_pages)
    total_skipped_empty += n_skipped

    print(f"{fname}: {n_before} halaman dimuat, {n_skipped} halaman kosong dilewati, {len(non_empty_pages)} halaman dipakai.")

print(f"\nTotal halaman terpakai (semua dokumen): {total_pages}")
print(f"Total halaman kosong dilewati (semua dokumen): {total_skipped_empty}")


C:\Users\Hype\AppData\Local\Temp\ipykernel_25412\826573647.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


W0712 07:18:17.658000 25412 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


W0712 07:18:17.761000 25412 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


PP Nomor 35 Tahun 2021.pdf: 56 halaman dimuat, 0 halaman kosong dilewati, 56 halaman dipakai.


PP Nomor 51 Tahun 2023.pdf: 27 halaman dimuat, 0 halaman kosong dilewati, 27 halaman dipakai.


PP Nomor 5 Tahun 2021.pdf: 739 halaman dimuat, 0 halaman kosong dilewati, 739 halaman dipakai.


UU Nomor 6 Tahun 2023.pdf: 1127 halaman dimuat, 0 halaman kosong dilewati, 1127 halaman dipakai.

Total halaman terpakai (semua dokumen): 1949
Total halaman kosong dilewati (semua dokumen): 0


## 7. Audit Kualitas Ekstraksi Teks

In [6]:
for fname, pages in raw_pages_by_file.items():
    assert len(pages) > 0, f"Tidak ada halaman non-kosong untuk {fname}"
    sample_text = pages[0].page_content.strip()
    assert len(sample_text) > 0, f"Halaman pertama {fname} kosong"
    preview = sample_text[:200].replace("\n", " ")
    print(f"--- {fname} (halaman pertama non-kosong) ---")
    print(preview)
    print()


--- PP Nomor 35 Tahun 2021.pdf (halaman pertama non-kosong) ---
SALINAN PRESIDEN REPUBLIK INDONESIA PERATURAN PEMERINTAH REPUBLIK INDONESIA NOMOR 35 TAHUN 2O2I TENTANG PERJANJIAN KERJA WAKTU TERTENTU, ALIH DAYA, WAKTU KERJA DAN WAKTU ISTIRAHAT, DAN PEMUTUSAN HUBUN

--- PP Nomor 51 Tahun 2023.pdf (halaman pertama non-kosong) ---
SALINAN PRESIDEN REPUBLTI( INDONESIA PERATURAN PEMERINTAH REPUBLIK INDONESIA NOMOR 51 TAHUN 2023 TENTANG PERUBAHAN ATAS PERATURAN PEMERINTAH NOMOR 36 TAHUN 2O21 TENTANG PENGUPAHAN DENGAN RAHMAT TUHAN 

--- PP Nomor 5 Tahun 2021.pdf (halaman pertama non-kosong) ---
LEMBARAN NEGARA  REPUBLIK INDONESIA  No.15, 2021 ADMINISTRASI. Perizinan Berusaha Berbasis  Risiko. Penyelenggaraan. (Penjelasan dalam  Tambahan Lembaran Negara Republik  Indonesia Nomor 6617)    PERA

--- UU Nomor 6 Tahun 2023.pdf (halaman pertama non-kosong) ---
SALINAN PRESIDEN NEPUBUK INDONESIA UNDANG-UNDANG REPUBLIK INDONESIA NOMOR 6 TAHUN 2023 TENTANG PENETAPAN PERATURAN PEMERINTAH PENGGANTI UNDA

## 8. Enrichment Metadata Regulasi

In [7]:
REGULATION_METADATA = {
    "PP Nomor 35 Tahun 2021.pdf": {
        "regulation_type": "Peraturan Pemerintah",
        "regulation_number": 35,
        "regulation_year": 2021,
        "official_title": "Perjanjian Kerja Waktu Tertentu, Alih Daya, Waktu Kerja dan Waktu Istirahat, dan Pemutusan Hubungan Kerja",
        "legal_topic": "ketenagakerjaan, PKWT, alih daya, waktu kerja, waktu istirahat, lembur, PHK",
    },
    "PP Nomor 51 Tahun 2023.pdf": {
        "regulation_type": "Peraturan Pemerintah",
        "regulation_number": 51,
        "regulation_year": 2023,
        "official_title": "Perubahan atas Peraturan Pemerintah Nomor 36 Tahun 2021 tentang Pengupahan",
        "legal_topic": "pengupahan dan upah minimum",
    },
    "PP Nomor 5 Tahun 2021.pdf": {
        "regulation_type": "Peraturan Pemerintah",
        "regulation_number": 5,
        "regulation_year": 2021,
        "official_title": "Penyelenggaraan Perizinan Berusaha Berbasis Risiko",
        "legal_topic": "perizinan berusaha berbasis risiko",
    },
    "UU Nomor 6 Tahun 2023.pdf": {
        "regulation_type": "Undang-Undang",
        "regulation_number": 6,
        "regulation_year": 2023,
        "official_title": "Penetapan Peraturan Pemerintah Pengganti Undang-Undang Nomor 2 Tahun 2022 tentang Cipta Kerja Menjadi Undang-Undang",
        "legal_topic": "Cipta Kerja dan perubahan regulasi lintas sektor",
    },
}

enriched_pages = []

for fname, pages in raw_pages_by_file.items():
    meta = REGULATION_METADATA[fname]
    source_path = resolved_pdf_paths[fname]
    for idx, page in enumerate(pages):
        page_number = page.metadata.get("page", idx) + 1  # PyPDFLoader page is 0-indexed
        page.metadata.update(meta)
        page.metadata["source_filename"] = fname
        page.metadata["source_path"] = source_path
        page.metadata["page_number"] = page_number
        page.metadata["source_id"] = f"{fname}::p{page_number}"
        enriched_pages.append(page)

print(f"Total halaman ter-enrich dengan metadata: {len(enriched_pages)}")
print("Contoh metadata halaman pertama:")
print(enriched_pages[0].metadata)


Total halaman ter-enrich dengan metadata: 1949
Contoh metadata halaman pertama:
{'producer': '', 'creator': 'Canon', 'creationdate': '2021-02-18T15:54:05+07:00', 'moddate': '2021-02-18T16:07:05+07:00', 'source': '.\\PP Nomor 35 Tahun 2021.pdf', 'total_pages': 56, 'page': 0, 'page_label': '1', 'regulation_type': 'Peraturan Pemerintah', 'regulation_number': 35, 'regulation_year': 2021, 'official_title': 'Perjanjian Kerja Waktu Tertentu, Alih Daya, Waktu Kerja dan Waktu Istirahat, dan Pemutusan Hubungan Kerja', 'legal_topic': 'ketenagakerjaan, PKWT, alih daya, waktu kerja, waktu istirahat, lembur, PHK', 'source_filename': 'PP Nomor 35 Tahun 2021.pdf', 'source_path': '.\\PP Nomor 35 Tahun 2021.pdf', 'page_number': 1, 'source_id': 'PP Nomor 35 Tahun 2021.pdf::p1'}


## 9. Parent/Child Chunking

In [8]:
import hashlib
from langchain_text_splitters import RecursiveCharacterTextSplitter

parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=PARENT_CHUNK_SIZE,
    chunk_overlap=PARENT_OVERLAP,
    separators=["\n\n", "\n", ". ", " "],
)

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHILD_CHUNK_SIZE,
    chunk_overlap=CHILD_OVERLAP,
    separators=["\n\n", "\n", ". ", " "],
)

def deterministic_id(*parts) -> str:
    raw = "||".join(str(p) for p in parts)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:24]

parent_documents = []
parent_docstore = {}

for page in enriched_pages:
    parent_texts = parent_splitter.split_text(page.page_content)
    for p_idx, p_text in enumerate(parent_texts):
        parent_id = deterministic_id(page.metadata["source_id"], "parent", p_idx)
        parent_meta = dict(page.metadata)
        parent_meta["parent_id"] = parent_id
        from langchain_core.documents import Document
        parent_doc = Document(page_content=p_text, metadata=parent_meta)
        parent_documents.append(parent_doc)
        parent_docstore[parent_id] = parent_doc

child_documents = []

for parent_doc in parent_documents:
    child_texts = child_splitter.split_text(parent_doc.page_content)
    parent_id = parent_doc.metadata["parent_id"]
    for c_idx, c_text in enumerate(child_texts):
        child_id = deterministic_id(parent_id, "child", c_idx)
        child_meta = dict(parent_doc.metadata)
        child_meta["child_id"] = child_id
        from langchain_core.documents import Document
        child_doc = Document(page_content=c_text, metadata=child_meta)
        child_documents.append(child_doc)

print(f"Jumlah parent documents: {len(parent_documents)}")
print(f"Jumlah child documents: {len(child_documents)}")


Jumlah parent documents: 1992
Jumlah child documents: 6067


## 10. Model Embedding (E5 multilingual)

In [9]:
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

class E5Embeddings(HuggingFaceEmbeddings):
    """Wrapper untuk model E5 yang mewajibkan prefix 'query: ' dan 'passage: '."""

    def embed_query(self, text: str):
        return super().embed_query(f"query: {text}")

    def embed_documents(self, texts):
        prefixed = [f"passage: {t}" for t in texts]
        return super().embed_documents(prefixed)

embedding_model = E5Embeddings(model_name=EMBEDDING_MODEL_ID)

_test_vector = embedding_model.embed_query("contoh pertanyaan uji coba")
EMBEDDING_DIM = len(_test_vector)

print(f"Model embedding: {EMBEDDING_MODEL_ID}")
print(f"Dimensi embedding: {EMBEDDING_DIM}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model embedding: intfloat/multilingual-e5-small
Dimensi embedding: 384


## 11. Vector Database (Chroma) — hanya menyimpan CHILD documents

> Catatan: `persist_directory` menggunakan folder sementara (`tempfile.mkdtemp()`) dan **dikecualikan dari ZIP submission final** karena bersifat lokal/regenerable.

In [10]:
import tempfile
from langchain_chroma import Chroma

CHROMA_PERSIST_DIR = tempfile.mkdtemp(prefix="pgabl_chroma_")
COLLECTION_NAME = "legal_child_chunks"

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model,
    persist_directory=CHROMA_PERSIST_DIR,
)

# Chroma membatasi ukuran batch per panggilan upsert (mis. ~5461). Untuk dataset child documents
# yang besar (mis. dari UU Nomor 6 Tahun 2023 yang berisi 1127 halaman), kita masukkan dokumen
# secara batch agar tidak melampaui batas tersebut.
try:
    _max_batch = vectorstore._client.get_max_batch_size()
except Exception:
    _max_batch = 4000

_batch_size = max(1, _max_batch - 1)
for _start in range(0, len(child_documents), _batch_size):
    vectorstore.add_documents(child_documents[_start:_start + _batch_size])

print(f"Jumlah child documents ter-index: {len(child_documents)}")
print(f"Collection name: {COLLECTION_NAME}")
print(f"Persist path (sementara, tidak disertakan di ZIP): {CHROMA_PERSIST_DIR}")

_sample_query = "uang lembur kerja"
_sample_results = vectorstore.similarity_search(_sample_query, k=1)
print("\nContoh hasil similarity_search untuk query:", _sample_query)
if _sample_results:
    r = _sample_results[0]
    print("source:", r.metadata.get("source_filename"), "| halaman:", r.metadata.get("page_number"))
    print("preview:", r.page_content[:200])


Jumlah child documents ter-index: 6067
Collection name: legal_child_chunks
Persist path (sementara, tidak disertakan di ZIP): C:\Users\Hype\AppData\Local\Temp\pgabl_chroma_rdlwf7a4

Contoh hasil similarity_search untuk query: uang lembur kerja
source: PP Nomor 35 Tahun 2021.pdf | halaman: 3
preview: perundang-undangan, termasuk tunjangan bagi
Pekerja/Buruh dan keluarganya atas suatu pekerjaan
dan/atau jasa yang telah atau akan dilakukan.
Waktu Kerja Lembur adalah waktu kerja yang melebihi
7 (tuju


## 12. Parent-Child Retriever

In [11]:
def get_parent_for_children(child_docs):
    """Memetakan child docs ke parent docs unik, mempertahankan urutan, dedup by parent_id."""
    seen_parent_ids = set()
    parents = []
    for child in child_docs:
        parent_id = child.metadata.get("parent_id")
        if parent_id and parent_id not in seen_parent_ids:
            seen_parent_ids.add(parent_id)
            parent_doc = parent_docstore.get(parent_id)
            if parent_doc is not None:
                parents.append(parent_doc)
    return parents

_demo_query = "berapa lama waktu istirahat kerja yang wajib diberikan?"
_demo_children = vectorstore.similarity_search(_demo_query, k=5)
_demo_parents = get_parent_for_children(_demo_children)

print(f"Demo query: {_demo_query}")
for child in _demo_children:
    parent_id = child.metadata.get("parent_id")
    print(
        f"child_id={child.metadata.get('child_id')} -> parent_id={parent_id} -> "
        f"source={child.metadata.get('source_filename')} -> halaman={child.metadata.get('page_number')} -> "
        f"preview={child.page_content[:80].strip()!r}"
    )

print(f"\nJumlah parent unik hasil mapping: {len(_demo_parents)}")


Demo query: berapa lama waktu istirahat kerja yang wajib diberikan?
child_id=ee5bec652e250d69ff6d56c0 -> parent_id=84ee0ff6d6bc0d30796dd8c2 -> source=UU Nomor 6 Tahun 2023.pdf -> halaman=559 -> preview='lembur dan Upah kerja lembur diatur dalam\nPeraturan Pemerintah.\n25. Ketentuan Pa'
child_id=d1a97bc7438a8a3e4b7216e7 -> parent_id=6745038e9b3b4d417f8e73c3 -> source=UU Nomor 6 Tahun 2023.pdf -> halaman=560 -> preview="I'RESIDEN\nREFTIEUK INDONESIA\n-550-\n(5) Selain waktu istirahat dan cuti sebagaima"
child_id=7a44d48fc63d7f021adee29f -> parent_id=0d2fb0b033264869fdc6c8bd -> source=PP Nomor 35 Tahun 2021.pdf -> halaman=15 -> preview='Perusahaan, atau Perjanjian Kerja Bersama.\nPasal22\nPengusaha yang mempekerjakan'
child_id=58d75d3f2e8a1f1a355db5b1 -> parent_id=84ee0ff6d6bc0d30796dd8c2 -> source=UU Nomor 6 Tahun 2023.pdf -> halaman=559 -> preview='setengah jam setelah bekerja selama 4 (empat)\njam terus-menerus, dan waktu istir'
child_id=ffeffb3ea02762bd1152594d -> parent_id=6745038e9b

## 13. BM25 Retriever (Keyword-based, level Parent Document)

In [12]:
import re
from rank_bm25 import BM25Okapi

def simple_tokenize(text: str):
    return re.findall(r"[a-z0-9]+", text.lower())

parent_texts_for_bm25 = [doc.page_content for doc in parent_documents]
tokenized_parent_corpus = [simple_tokenize(t) for t in parent_texts_for_bm25]
bm25_index = BM25Okapi(tokenized_parent_corpus)

def bm25_search(query: str, k: int = CANDIDATE_K):
    tokenized_query = simple_tokenize(query)
    scores = bm25_index.get_scores(tokenized_query)
    ranked_idx = np.argsort(scores)[::-1][:k]
    results = []
    for idx in ranked_idx:
        results.append((parent_documents[idx], float(scores[idx])))
    return results

_bm25_demo = bm25_search("uang lembur", k=5)
print("Contoh hasil BM25 untuk 'uang lembur':")
for doc, score in _bm25_demo:
    print(f" score={score:.4f} | source={doc.metadata.get('source_filename')} | halaman={doc.metadata.get('page_number')} | preview={doc.page_content[:80].strip()!r}")


Contoh hasil BM25 untuk 'uang lembur':
 score=14.1956 | source=PP Nomor 35 Tahun 2021.pdf | halaman=18 | preview='PRESiDEN\nREPUBLIK INDONESI,A.\n-18-\n(21 Perintah dan persetujuan sebagaimana dima'
 score=13.9911 | source=PP Nomor 35 Tahun 2021.pdf | halaman=44 | preview='FRESIDEN\nREPUBLIK INDONESIA\n-2-\nd. pelindungan Pekerja/Buruh dan perizinan berus'
 score=10.4715 | source=PP Nomor 35 Tahun 2021.pdf | halaman=17 | preview='PRESIDEN\nREPUBLIK INDONESIA\n-t7-\nBagian Ketiga\nWaktu Kerja Lembur\nPasal 26\n(1) W'
 score=10.3475 | source=PP Nomor 35 Tahun 2021.pdf | halaman=19 | preview='PRESIDEN\nREPUELIK INDONESIA\n-L9-\na. untuk ja- kerja lembur pertama sebesar\n1,5 ('
 score=10.0672 | source=PP Nomor 35 Tahun 2021.pdf | halaman=3 | preview='PRESIDEN\nREPUBLIK INDONESIA\n-3-\n5\nb. usaha-usaha sosial dan usaha-usaha lain yan'


## 14. Ensemble Retriever — Weighted Reciprocal Rank Fusion (BM25 + Semantic)

In [13]:
def _reciprocal_rank_scores(ranked_items, weight, rrf_k: int = 60):
    """ranked_items: list of (parent_id, ...) in rank order (best first). Returns dict parent_id -> weighted RRF score."""
    scores = {}
    for rank, parent_id in enumerate(ranked_items, start=1):
        scores[parent_id] = scores.get(parent_id, 0.0) + weight * (1.0 / (rrf_k + rank))
    return scores

def ensemble_retrieve(query: str, candidate_k: int = CANDIDATE_K):
    # BM25 branch (already at parent level)
    bm25_results = bm25_search(query, k=candidate_k)
    bm25_parent_ids_ranked = [doc.metadata["parent_id"] for doc, _ in bm25_results]

    # Semantic branch: search children, then map to parents
    semantic_children = vectorstore.similarity_search(query, k=candidate_k * 2)
    semantic_parents = get_parent_for_children(semantic_children)
    semantic_parent_ids_ranked = [doc.metadata["parent_id"] for doc in semantic_parents]

    bm25_scores = _reciprocal_rank_scores(bm25_parent_ids_ranked, BM25_WEIGHT)
    semantic_scores = _reciprocal_rank_scores(semantic_parent_ids_ranked, SEMANTIC_WEIGHT)

    fused_scores = {}
    for pid, s in bm25_scores.items():
        fused_scores[pid] = fused_scores.get(pid, 0.0) + s
    for pid, s in semantic_scores.items():
        fused_scores[pid] = fused_scores.get(pid, 0.0) + s

    ranked_parent_ids = sorted(fused_scores.keys(), key=lambda pid: fused_scores[pid], reverse=True)

    results = []
    for pid in ranked_parent_ids[:candidate_k]:
        doc = parent_docstore.get(pid)
        if doc is not None:
            results.append((doc, fused_scores[pid]))
    return results

_ensemble_demo = ensemble_retrieve("uang lembur kerja", candidate_k=CANDIDATE_K)
print(f"Jumlah kandidat unik hasil ensemble: {len(_ensemble_demo)}")
for doc, score in _ensemble_demo:
    print(f" fused_score={score:.5f} | source={doc.metadata.get('source_filename')} | halaman={doc.metadata.get('page_number')}")


Jumlah kandidat unik hasil ensemble: 10
 fused_score=0.01608 | source=PP Nomor 35 Tahun 2021.pdf | halaman=18
 fused_score=0.01590 | source=PP Nomor 35 Tahun 2021.pdf | halaman=3
 fused_score=0.01572 | source=PP Nomor 35 Tahun 2021.pdf | halaman=17
 fused_score=0.01565 | source=UU Nomor 6 Tahun 2023.pdf | halaman=559
 fused_score=0.01548 | source=PP Nomor 35 Tahun 2021.pdf | halaman=19
 fused_score=0.01528 | source=PP Nomor 35 Tahun 2021.pdf | halaman=44
 fused_score=0.01485 | source=PP Nomor 35 Tahun 2021.pdf | halaman=21
 fused_score=0.00909 | source=UU Nomor 6 Tahun 2023.pdf | halaman=558
 fused_score=0.00896 | source=PP Nomor 35 Tahun 2021.pdf | halaman=20
 fused_score=0.00857 | source=PP Nomor 35 Tahun 2021.pdf | halaman=56


## 15. Metadata Filtering Berdasarkan Regex Query

In [14]:
import re

_REGEX_PP_UU_SLASH = re.compile(r"(PP|UU)\s*(\d+)\s*/\s*(\d{4})", re.IGNORECASE)
_REGEX_PP_UU_FORMAL = re.compile(
    r"(Peraturan Pemerintah|Undang-Undang|PP|UU)\s*(?:Nomor|No\.?)?\s*(\d+)\s*Tahun\s*(\d{4})",
    re.IGNORECASE,
)

_TYPE_MAP = {"pp": "Peraturan Pemerintah", "uu": "Undang-Undang"}

def parse_regulation_filter(query: str):
    """Mengekstrak filter regulasi {regulation_type, regulation_number, regulation_year} dari query.
    Mengembalikan dict penuh/sebagian, atau None jika tidak ada indikasi regulasi spesifik."""
    for pattern in (_REGEX_PP_UU_SLASH, _REGEX_PP_UU_FORMAL):
        match = pattern.search(query)
        if match:
            type_token, number, year = match.groups()
            regulation_type = _TYPE_MAP.get(type_token.strip().lower(), type_token.strip())
            return {
                "regulation_type": regulation_type,
                "regulation_number": int(number),
                "regulation_year": int(year),
            }

    # Deteksi generik tanpa nomor/tahun spesifik
    lowered = query.lower()
    if "peraturan pemerintah" in lowered or re.search(r"\bpp\b", lowered):
        return {"regulation_type": "Peraturan Pemerintah"}
    if "undang-undang" in lowered or re.search(r"\buu\b", lowered):
        return {"regulation_type": "Undang-Undang"}

    return None


def apply_metadata_filter(candidates, filter_dict, min_results: int = 5, query_for_refetch: str = None, candidate_k: int = CANDIDATE_K):
    """candidates: list of (doc, score). Memfilter berdasarkan metadata jika filter_dict tidak kosong."""
    if not filter_dict:
        return candidates

    def matches(doc):
        meta = doc.metadata
        for key, value in filter_dict.items():
            if meta.get(key) != value:
                return False
        return True

    filtered = [(doc, score) for doc, score in candidates if matches(doc)]

    if len(filtered) < min_results and query_for_refetch:
        wider_candidates = ensemble_retrieve(query_for_refetch, candidate_k=candidate_k * 3)
        filtered = [(doc, score) for doc, score in wider_candidates if matches(doc)]

    return filtered


# Demo: query spesifik PP 51/2023
_demo_query_filter = "Berdasarkan PP Nomor 51 Tahun 2023, apa fokus perubahan aturan pengupahan?"
_active_filter = parse_regulation_filter(_demo_query_filter)
print("Filter aktif terdeteksi:", _active_filter)

_before_candidates = ensemble_retrieve(_demo_query_filter, candidate_k=CANDIDATE_K)
print("Jumlah kandidat sebelum filter:", len(_before_candidates))

_after_candidates = apply_metadata_filter(
    _before_candidates, _active_filter, min_results=5, query_for_refetch=_demo_query_filter
)
print("Jumlah kandidat setelah filter:", len(_after_candidates))

assert _active_filter is not None
assert _active_filter.get("regulation_number") == 51
assert _active_filter.get("regulation_year") == 2023
print("Filter PP 51/2023 terbukti terdeteksi dengan benar (regulation_number=51, regulation_year=2023).")


Filter aktif terdeteksi: {'regulation_type': 'Peraturan Pemerintah', 'regulation_number': 51, 'regulation_year': 2023}
Jumlah kandidat sebelum filter: 10
Jumlah kandidat setelah filter: 4
Filter PP 51/2023 terbukti terdeteksi dengan benar (regulation_number=51, regulation_year=2023).


## 16. HyDE (Hypothetical Document Embeddings) — Query Expansion

Catatan: HyDE bersifat *internal query expansion* untuk memperluas retrieval — jawaban hipotetis **tidak pernah ditampilkan ke pengguna sebagai fakta**. Fungsi ini menerima `generate_fn` yang baru di-bind setelah model GRPO dimuat (Bagian 21).

In [15]:
def generate_hyde_answers(question: str, generate_fn, n: int = 2):
    """Menghasilkan n jawaban hipotetis menggunakan generate_fn, dengan temperature/seed berbeda tiap panggilan.
    HANYA untuk ekspansi query internal, TIDAK ditampilkan ke pengguna sebagai fakta."""
    hyde_prompt_template = (
        "Anda adalah asisten hukum. Tuliskan jawaban singkat hipotetis (asumsi, boleh tidak sepenuhnya akurat) "
        "untuk pertanyaan berikut, seolah-olah Anda mengetahui isi regulasi ketenagakerjaan Indonesia terkait.\n\n"
        f"Pertanyaan: {question}\n\nJawaban hipotetis singkat:"
    )

    hypothetical_answers = []
    for i in range(n):
        temperature = 0.6 + (i * 0.2)
        seed = SEED + i
        answer = generate_fn(hyde_prompt_template, temperature=temperature, seed=seed)
        hypothetical_answers.append(answer)
        print(f"[HyDE internal, tidak ditampilkan ke user] percobaan {i+1} (temp={temperature}, seed={seed}) dihasilkan.")

    return hypothetical_answers


## 17. Perakitan Kandidat (Minimal 5 Unik Sebelum Reranking)

In [16]:
def deduplicate_candidates(candidate_lists):
    """candidate_lists: list of list[(doc, score)]. Menghasilkan list doc unik (dedup by parent_id), mempertahankan urutan skor tertinggi pertama kali muncul."""
    seen_ids = set()
    unique_docs = []
    for candidates in candidate_lists:
        for doc, score in candidates:
            pid = doc.metadata.get("parent_id")
            if pid not in seen_ids:
                seen_ids.add(pid)
                unique_docs.append(doc)
    return unique_docs


def assemble_candidates(question: str, hyde_answers, candidate_k: int = CANDIDATE_K):
    all_queries = [question] + list(hyde_answers)
    candidate_lists = [ensemble_retrieve(q, candidate_k=candidate_k) for q in all_queries]
    unique_docs = deduplicate_candidates(candidate_lists)
    return unique_docs


# Demo tanpa HyDE nyata (model belum dimuat) -- gunakan hanya query asli untuk demonstrasi assembly count di titik ini
_demo_candidates_preview = deduplicate_candidates([ensemble_retrieve("uang lembur kerja", candidate_k=CANDIDATE_K)])
print(f"[Demo pra-HyDE] Jumlah kandidat unik (hanya query asli): {len(_demo_candidates_preview)}")
print("Catatan: perakitan kandidat penuh (question + 2 HyDE) akan didemonstrasikan pada Bagian 21-24 setelah model GRPO dimuat.")


[Demo pra-HyDE] Jumlah kandidat unik (hanya query asli): 10


Catatan: perakitan kandidat penuh (question + 2 HyDE) akan didemonstrasikan pada Bagian 21-24 setelah model GRPO dimuat.


## 18. Cross-Encoder Reranker

In [17]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder(RERANKER_MODEL_ID)

def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def rerank_candidates(question: str, candidate_docs, top_k: int = TOP_K_FINAL):
    if not candidate_docs:
        return []

    pairs = [(question, doc.page_content) for doc in candidate_docs]
    raw_scores = cross_encoder.predict(pairs)
    normalized_scores = _sigmoid(np.array(raw_scores))

    scored = list(zip(candidate_docs, raw_scores, normalized_scores))
    scored.sort(key=lambda x: x[2], reverse=True)

    top_results = scored[:top_k]

    print(f"{'rank':<5}{'source_filename':<30}{'page':<6}{'raw_score':<12}{'norm_score':<12}preview")
    for rank, (doc, raw_s, norm_s) in enumerate(top_results, start=1):
        preview = doc.page_content[:60].replace("\n", " ").strip()
        print(f"{rank:<5}{doc.metadata.get('source_filename',''):<30}{doc.metadata.get('page_number',''):<6}{float(raw_s):<12.4f}{float(norm_s):<12.4f}{preview}")

    return top_results

_rerank_demo = rerank_candidates("uang lembur kerja", _demo_candidates_preview, top_k=TOP_K_FINAL)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

rank source_filename               page  raw_score   norm_score  preview
1    PP Nomor 35 Tahun 2021.pdf    3     0.5782      0.6407      PRESIDEN REPUBLIK INDONESIA -3- 5 b. usaha-usaha sosial dan
2    PP Nomor 35 Tahun 2021.pdf    19    -0.5646     0.3625      PRESIDEN REPUELIK INDONESIA -L9- a. untuk ja- kerja lembur p
3    PP Nomor 35 Tahun 2021.pdf    20    -0.7760     0.3152      PRESIDEN REPUBLIK INDONESIA -20- b. jam kesembilan, dibayar


## 19. Skor Relevansi Top-1 & Threshold Routing

In [18]:
def decide_routing(reranked_results):
    """reranked_results: list of (doc, raw_score, normalized_score), diurutkan menurun.
    Mengembalikan (path, top1_score) dengan path in {'LOCAL_RAG', 'WEB_FALLBACK'}."""
    if not reranked_results:
        return "WEB_FALLBACK", 0.0

    top1_score = float(reranked_results[0][2])
    path = "LOCAL_RAG" if top1_score >= RERANK_THRESHOLD else "WEB_FALLBACK"
    return path, top1_score

_demo_path, _demo_top1 = decide_routing(_rerank_demo)
print(f"Top-1 normalized_relevance_score: {_demo_top1:.4f}")
print(f"Threshold: {RERANK_THRESHOLD}")
print(f"Path terpilih: {_demo_path}")


Top-1 normalized_relevance_score: 0.6407
Threshold: 0.5
Path terpilih: LOCAL_RAG


## 20. Fallback Pencarian Web (DuckDuckGo)

> Hasil web **tidak pernah** ditambahkan ke vector DB persisten — hanya dipakai sebagai konteks sementara untuk satu kali generasi jawaban.

In [19]:
from duckduckgo_search import DDGS

def duckduckgo_fallback(question: str, max_results: int = 5):
    """Mengembalikan list dict {title, body, href}. Bias ke sumber hukum resmi Indonesia via query tambahan.
    TIDAK menambahkan hasil ke vector DB persisten."""
    queries_to_try = [
        question,
        f"{question} site:peraturan.bpk.go.id OR jdih",
    ]

    collected = []
    try:
        with DDGS() as ddgs:
            for q in queries_to_try:
                results = ddgs.text(q, max_results=max_results)
                for r in results:
                    collected.append({
                        "title": r.get("title", ""),
                        "body": r.get("body", ""),
                        "href": r.get("href", ""),
                    })
                if len(collected) >= max_results:
                    break
    except Exception as e:
        print(f"[duckduckgo_fallback] Pencarian web gagal: {e}")
        return {"status": "unavailable", "reason": str(e), "results": []}

    if not collected:
        return {"status": "unavailable", "reason": "no results", "results": []}

    return {"status": "ok", "reason": None, "results": collected[:max_results]}


def format_web_context(web_response) -> str:
    if web_response["status"] != "ok":
        return "(Pencarian web tidak tersedia saat ini.)"
    parts = []
    for r in web_response["results"]:
        parts.append(f"Judul: {r['title']}\nRingkasan: {r['body']}\nSumber: {r['href']}")
    return "\n\n".join(parts)

print("Fungsi duckduckgo_fallback siap. Hasil web TIDAK ditambahkan ke vector DB persisten.")


Fungsi duckduckgo_fallback siap. Hasil web TIDAK ditambahkan ke vector DB persisten.


## 21. Memuat Model GRPO (Inference Only)

> **Catatan CPU-only:** Model dimuat dengan `transformers.AutoModelForCausalLM` (bukan `unsloth.FastLanguageModel` 4-bit, yang mensyaratkan GPU/CUDA). Model yang dimuat adalah model GRPO publik hasil notebook GRPO sebelumnya (`HuggingFaceTB/SmolLM2-135M-Instruct` + adapter LoRA GRPO yang sudah di-merge), yang telah diunggah ke Hugging Face Hub sehingga proses pemuatan di sini sepenuhnya nyata (bukan simulasi/placeholder).


In [20]:
from transformers import AutoModelForCausalLM, AutoTokenizer

grpo_tokenizer = AutoTokenizer.from_pretrained(GRPO_MODEL_ID, token=HF_TOKEN)
if grpo_tokenizer.pad_token is None:
    grpo_tokenizer.pad_token = grpo_tokenizer.eos_token

grpo_model = AutoModelForCausalLM.from_pretrained(
    GRPO_MODEL_ID,
    dtype=torch.float32,
    token=HF_TOKEN,
)
grpo_model.eval()

print(f"Model GRPO berhasil dimuat untuk inference: {GRPO_MODEL_ID}")

def generate_fn(prompt: str, temperature: float = 0.7, seed: int = SEED, max_new_tokens: int = 512) -> str:
    """Wrapper generate untuk model GRPO, dipakai oleh HyDE dan generasi jawaban final."""
    set_seed(seed)
    inputs = grpo_tokenizer(prompt, return_tensors="pt").to(grpo_model.device)
    with torch.no_grad():
        output_ids = grpo_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=grpo_tokenizer.eos_token_id,
        )
    decoded = grpo_tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return decoded.strip()

print("generate_fn siap digunakan.")

# Demo HyDE nyata memakai model yang sudah dimuat
_demo_question_hyde = "Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?"
_demo_hyde_answers = generate_hyde_answers(_demo_question_hyde, generate_fn, n=2)
print(f"Jumlah jawaban HyDE dihasilkan: {len(_demo_hyde_answers)}")
print("Berbeda satu sama lain:", _demo_hyde_answers[0] != _demo_hyde_answers[1] if len(_demo_hyde_answers) == 2 else False)


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Model GRPO berhasil dimuat untuk inference: galiihajiip/pgabl-legal-grpo-smollm2-135m-galih
generate_fn siap digunakan.


[HyDE internal, tidak ditampilkan ke user] percobaan 1 (temp=0.6, seed=42) dihasilkan.


[HyDE internal, tidak ditampilkan ke user] percobaan 2 (temp=0.8, seed=43) dihasilkan.
Jumlah jawaban HyDE dihasilkan: 2
Berbeda satu sama lain: True


## 22. Template Prompt (Local RAG & Web Fallback)

In [21]:
LOCAL_RAG_PROMPT = """Anda adalah asisten hukum ketenagakerjaan Indonesia. Jawab pertanyaan HANYA berdasarkan konteks yang diberikan di bawah ini.
Jawab dalam Bahasa Indonesia. Jangan berspekulasi. Jika konteks tidak cukup untuk menjawab, katakan secara eksplisit bahwa informasi tidak tersedia dalam konteks.
Jangan mengarang nomor pasal atau sumber apa pun — sistem akan menambahkan kutipan sumber secara terpisah setelah jawaban Anda.

Mulailah jawaban Anda dengan tepat satu blok <think>...</think> berisi penalaran singkat, lalu berikan jawaban akhir setelahnya.

Konteks:
{context}

Pertanyaan: {question}

Jawaban:"""

WEB_FALLBACK_PROMPT = """Anda adalah asisten hukum ketenagakerjaan Indonesia. Pertanyaan pengguna tidak dapat dijawab dari basis dokumen regulasi lokal, sehingga sistem menggunakan hasil pencarian web di bawah ini sebagai konteks TAMBAHAN.
PERLAKUKAN hasil web ini sebagai TIDAK TERVERIFIKASI (unverified) — jangan menyatakannya sebagai kepastian hukum mutlak.
Jawab dalam Bahasa Indonesia. Jangan berspekulasi di luar konteks yang diberikan. Jika konteks tidak cukup, katakan secara eksplisit.
Jangan mengarang nomor pasal atau sumber apa pun — sistem akan menambahkan kutipan sumber secara terpisah setelah jawaban Anda.

Mulailah jawaban Anda dengan tepat satu blok <think>...</think> berisi penalaran singkat, lalu berikan jawaban akhir setelahnya.

Konteks (hasil pencarian web, TIDAK TERVERIFIKASI):
{context}

Pertanyaan: {question}

Jawaban:"""

print("Template prompt LOCAL_RAG_PROMPT dan WEB_FALLBACK_PROMPT siap.")


Template prompt LOCAL_RAG_PROMPT dan WEB_FALLBACK_PROMPT siap.


## 23. Pembangun Kutipan (Citations)

In [22]:
def _regulation_label(meta: dict) -> str:
    reg_type = meta.get("regulation_type", "")
    reg_number = meta.get("regulation_number", "")
    reg_year = meta.get("regulation_year", "")
    if reg_type == "Peraturan Pemerintah":
        return f"PP Nomor {reg_number} Tahun {reg_year}"
    if reg_type == "Undang-Undang":
        return f"UU Nomor {reg_number} Tahun {reg_year}"
    return reg_type or "Sumber tidak diketahui"


def build_citations(retrieved_docs) -> str:
    seen = set()
    lines = []
    for doc in retrieved_docs:
        meta = doc.metadata
        label = _regulation_label(meta)
        page = meta.get("page_number", "?")
        key = (label, page)
        if key not in seen:
            seen.add(key)
            lines.append(f"- {label}, halaman {page}")
    return "\n".join(lines) if lines else "- (tidak ada sumber lokal yang dikutip)"


def build_web_citations(web_results) -> str:
    seen = set()
    lines = []
    for r in web_results:
        key = r.get("href", "")
        if key and key not in seen:
            seen.add(key)
            lines.append(f"- {r.get('title', '(tanpa judul)')} \u2014 {r.get('href', '')}")
    return "\n".join(lines) if lines else "- (tidak ada sumber web yang dikutip)"

print("Fungsi build_citations dan build_web_citations siap.")


Fungsi build_citations dan build_web_citations siap.


## 24. Pipeline Utama — `answer_legal_question`

In [23]:
def answer_legal_question(question: str) -> str:
    try:
        if not question or not question.strip():
            return "**Error:** Pertanyaan tidak boleh kosong."

        question = question.strip()

        # 1. Parse metadata filter
        active_filter = parse_regulation_filter(question)

        # 2. HyDE query expansion (internal, tidak ditampilkan sebagai fakta)
        hyde_answers = generate_hyde_answers(question, generate_fn, n=2)

        # 3. Ensemble retrieve untuk original + 2 HyDE queries, gabung & dedup (>=5 unik jika tersedia)
        candidate_docs = assemble_candidates(question, hyde_answers, candidate_k=CANDIDATE_K)
        candidate_scored = [(doc, 0.0) for doc in candidate_docs]

        # 4. Terapkan filter metadata bila ada
        if active_filter:
            candidate_scored = apply_metadata_filter(
                candidate_scored, active_filter, min_results=5, query_for_refetch=question, candidate_k=CANDIDATE_K
            )
        filtered_docs = [doc for doc, _ in candidate_scored] if candidate_scored else candidate_docs

        if not filtered_docs:
            filtered_docs = candidate_docs

        # 5. Cross-encoder rerank
        reranked = rerank_candidates(question, filtered_docs, top_k=TOP_K_FINAL)
        path, top1_score = decide_routing(reranked)

        debug_line = f"_debug: top1_score={top1_score:.4f}, threshold={RERANK_THRESHOLD}, path={path}_"

        if path == "LOCAL_RAG":
            top_docs = [doc for doc, _, _ in reranked]
            context = "\n\n---\n\n".join(doc.page_content for doc in top_docs)
            prompt = LOCAL_RAG_PROMPT.format(context=context, question=question)
            generation = generate_fn(prompt, temperature=0.3, seed=SEED)
            citations = build_citations(top_docs)
            answer_md = (
                f"{generation}\n\n"
                f"**Sumber:**\n{citations}\n\n"
                f"{debug_line}"
            )
        else:
            web_response = duckduckgo_fallback(question, max_results=5)
            web_context = format_web_context(web_response)
            prompt = WEB_FALLBACK_PROMPT.format(context=web_context, question=question)
            generation = generate_fn(prompt, temperature=0.3, seed=SEED)
            web_citations = build_web_citations(web_response.get("results", []))
            answer_md = (
                f"{generation}\n\n"
                f"**Sumber Web (tidak terverifikasi):**\n{web_citations}\n\n"
                f"{debug_line}"
            )

        return answer_md

    except Exception as e:
        return f"**Terjadi kesalahan pada pipeline RAG:** `{e}`\n\nJawaban tidak dapat dibuat secara aman; silakan coba lagi."

print("Fungsi answer_legal_question siap digunakan.")


Fungsi answer_legal_question siap digunakan.


## 25. Uji Wajib #1 — Pertanyaan Uang Lembur

In [24]:
test1_question = "Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?"
test1_result = answer_legal_question(test1_question)
print(test1_result)


[HyDE internal, tidak ditampilkan ke user] percobaan 1 (temp=0.6, seed=42) dihasilkan.


[HyDE internal, tidak ditampilkan ke user] percobaan 2 (temp=0.8, seed=43) dihasilkan.


rank source_filename               page  raw_score   norm_score  preview
1    PP Nomor 35 Tahun 2021.pdf    19    -3.6974     0.0242      PRESIDEN REPUELIK INDONESIA -L9- a. untuk ja- kerja lembur p
2    PP Nomor 35 Tahun 2021.pdf    18    -4.4053     0.0121      PRESiDEN REPUBLIK INDONESI,A. -18- (21 Perintah dan persetuj
3    PP Nomor 35 Tahun 2021.pdf    3     -4.4640     0.0114      PRESIDEN REPUBLIK INDONESIA -3- 5 b. usaha-usaha sosial dan


C:\Users\Hype\AppData\Local\Temp\ipykernel_25412\2373499675.py:13: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?

Sumber: https://narabahasa.id/artikel/linguistik-umum/perjalanan-panjang-pronomina-saya/

Judul: Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?

Sumber: https://narabahasa.id/artikel/linguistik-umum/perjalanan-panjang-pronomina-saya/

Judul: Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?

Sumber: https://narabahasa.id/artikel/linguistik-umum/perjalanan-panjang-pronomina-saya/

Judul: Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?

Sumber: https://narabahasa.id/artikel/linguistik-umum/perjalanan-panjang-pronomina-saya/

Judul: Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?

Sumber: https://narabahasa.id/artikel/linguistik-umum/perjalanan-panjang-pronomina-saya/

Judul: Saya staf admin

## 26. Uji Wajib #2 — Filter Metadata PP 51/2023

In [25]:
test2_question = "Berdasarkan PP Nomor 51 Tahun 2023, apa fokus perubahan aturan pengupahan?"
test2_parsed_filter = parse_regulation_filter(test2_question)
test2_result = answer_legal_question(test2_question)
print(test2_result)

assert test2_parsed_filter is not None
assert test2_parsed_filter.get("regulation_number") == 51
assert test2_parsed_filter.get("regulation_year") == 2023
print("\nAssertion filter PP 51/2023 lolos: regulation_number=51, regulation_year=2023.")


[HyDE internal, tidak ditampilkan ke user] percobaan 1 (temp=0.6, seed=42) dihasilkan.


[HyDE internal, tidak ditampilkan ke user] percobaan 2 (temp=0.8, seed=43) dihasilkan.


rank source_filename               page  raw_score   norm_score  preview
1    PP Nomor 51 Tahun 2023.pdf    17    8.8445      0.9999      I FRES!E}EN REPUBLIK INDONESIA PENJELASAN ATAS PERATURAN PEM
2    PP Nomor 51 Tahun 2023.pdf    1     8.2735      0.9997      SALINAN PRESIDEN REPUBLTI( INDONESIA PERATURAN PEMERINTAH RE
3    PP Nomor 51 Tahun 2023.pdf    18    -1.6787     0.1573      FRESIDET{ RTPUBLII( INDONESIA -2- II Kebijakan pengupahan ya


---

PEMERITAN PEMERINTAH REPUBLIK INDONESIA
PENJELASAN
ATAS
PERATURAN PEMERINTAH PEMERINTAH NOMOR 36 Tahun 2O21
TENTANG
PERUBAHAN ATAS PERATURAN PEMERINTAH NOMOR 36 TAHUN 2O21
TENTANG PENGUPAHAN
DENGAN RAHMAT TUHAN YANG MAHA ESA
PEMERITAN PEMERINTAH REPUBLIK INDONESIA,
Menimbang a. bahwa untuk melaksanakan ketentuan
Pemerintah Pengganti Undang-Undang Nomor 2
Tahun 2023 tentang Cipta Kerja Menjadi Undang-Undang
perlu dilakukan perubahan mengenai ketentuan
Pemerintah Pengganti Undang-Undang Nomor 2
Tahun 2022 tentang Cipta Kerja Menjadi Undang-Undang
perlu dilakukan perubahan mengenai ketentuan
Pemerintah Pengganti Undang-Undang Nomor 2
Tahun 2021 tentang Cipta Kerja Menjadi Undang-Undang
perlu dilakukan perubahan mengenai ketentuan
Pemerintah Pengganti Undang-Undang Nomor 2
Tahun 2020 tentang Cipta Kerja Menjadi Undang-Undang
perlu dilakukan perubahan mengenai ketentuan
Pemerintah Pengganti Undang-Undang Nomor 2
Tahun 2019 tentang Cipta Kerja Menjadi Undang-Undang
perlu dilakukan perub

## 27. Uji Wajib #3 — Fallback Web (Pertanyaan di Luar Domain)

In [26]:
test3_question = "Bagaimana aturan pajak penghasilan pribadi di Jepang untuk tahun 2024?"
test3_result = answer_legal_question(test3_question)
print(test3_result)

if "path=WEB_FALLBACK" in test3_result:
    print("\nOK: path WEB_FALLBACK terpilih sesuai ekspektasi.")
else:
    print("\nWARN: path yang terpilih bukan WEB_FALLBACK. Skor relevansi runtime mungkin berbeda dari ekspektasi; periksa debug line di atas.")


[HyDE internal, tidak ditampilkan ke user] percobaan 1 (temp=0.6, seed=42) dihasilkan.


[HyDE internal, tidak ditampilkan ke user] percobaan 2 (temp=0.8, seed=43) dihasilkan.


rank source_filename               page  raw_score   norm_score  preview
1    UU Nomor 6 Tahun 2023.pdf     1065  -2.3108     0.0902      PRESTDEN REPUBLIK INDONESIA -318- Ayat (5) Pada prinsipnya p
2    UU Nomor 6 Tahun 2023.pdf     1064  -2.7743     0.0587      PRESIDEN REPUBLIK INDONESIA -3t7 - Ketentuan ini tidak diter
3    PP Nomor 5 Tahun 2021.pdf     314   -2.9259     0.0509      2021, No.15 -314-  huruf a; dan/atau  b. tidak memberikan ak


C:\Users\Hype\AppData\Local\Temp\ipykernel_25412\2373499675.py:13: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Bagaimana dalam konteks TAMBAHAN

Konteks (hasil pencarian web, TIDAK TERVERIFIKASI):
Judul: Arti kata "bagaimana" menjadi pajak penghasilan pribadi di Jepang untuk tahun 2024?
Ringkasan: Sep 26, 2024 · Bagaimana pajak penghasilan pribadi di Jepang untuk tahun 2024.
Sumber: https://kbbi.cari.co/bagaimana

Judul: Arti kata "bagaimana" menjadi pajak penghasilan pribadi di Jepang untuk tahun 2024?
Sumber: Sep 26, 2024 · Bagaimana pajak penghasilan pribadi di Jepang untuk tahun 2024.
Sumber: https://kbbi.cari.co/bagaimana

Judul: Arti kata "bagaimana" menjadi pajak penghasilan pribadi di Jepang untuk tahun 2024?
Sumber: Sep 26, 2024 · Bagaimana pajak penghasilan pribadi di Jepang untuk tahun 2024.
Sumber: https://kbbi.cari.co/bagaimana

Judul: Arti kata "bagaimana" menjadi pajak penghasilan pribadi di Jepang untuk tahun 2024?
Sumber: Sep 26, 2024 · Bagaimana pajak penghasilan pribadi di Jepang untuk tahun 2024.
Sumber: https://kbbi.cari.co/bagaimana

Judul: Arti kata "bagaimana" menjadi pa

## 28. Antarmuka Gradio

In [27]:
import gradio as gr

interface = gr.Interface(
    fn=answer_legal_question,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Tulis pertanyaan seputar ketenagakerjaan, pengupahan, perizinan, atau cipta kerja...",
    ),
    outputs=gr.Markdown(),
    title="Asisten Legal RAG - PGABL",
    description=(
        "Asisten tanya-jawab regulasi ketenagakerjaan Indonesia (PP 35/2021, PP 51/2023, PP 5/2021, UU 6/2023) "
        "menggunakan model generatif hasil fine-tuning GRPO sebagai satu-satunya generator jawaban. "
        "Sistem otomatis beralih ke pencarian web bila pertanyaan berada di luar cakupan dokumen."
    ),
    examples=[
        "Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?",
        "Berdasarkan PP Nomor 51 Tahun 2023, apa fokus perubahan aturan pengupahan?",
    ],
)


In [28]:
# Catatan: interface.launch() dinonaktifkan otomatis pada eksekusi non-interaktif (nbconvert/CI)
# agar notebook tidak hang menunggu server. Set RUN_GRADIO_LAUNCH=True untuk menjalankan UI secara interaktif.
RUN_GRADIO_LAUNCH = True
if RUN_GRADIO_LAUNCH:
    interface.launch(debug=False)
else:
    print("Gradio interface berhasil dibangun (objek 'interface'). Launch dilewati pada mode eksekusi otomatis.")


C:\Users\Hype\AppData\Local\Programs\Python\Python312\Lib\site-packages\websockets\legacy\__init__.py:6: DeprecationWarning: websockets.legacy is deprecated; see https://websockets.readthedocs.io/en/stable/howto/upgrade.html for upgrade instructions
  warnings.warn(  # deprecated in 14.0 - 2024-11-09
C:\Users\Hype\AppData\Local\Programs\Python\Python312\Lib\site-packages\uvicorn\protocols\websockets\websockets_impl.py:17: DeprecationWarning: websockets.server.WebSocketServerProtocol is deprecated
  from websockets.server import WebSocketServerProtocol


* Running on local URL:  http://127.0.0.1:7860


* To create a public link, set `share=True` in `launch()`.


## 29. Checklist Verifikasi Akhir (Programatik)

In [29]:
checklist_results = []

def check(name, condition):
    status = "PASS" if condition else "FAIL"
    checklist_results.append((name, status))
    print(f"[{status}] {name}")

check("Jumlah parent documents > 0", len(parent_documents) > 0)
check("Jumlah child documents > 0", len(child_documents) > 0)
check("Dimensi embedding > 0", EMBEDDING_DIM > 0)
check(
    "Parsing filter PP 51/2023 sesuai (regulation_number=51, regulation_year=2023)",
    test2_parsed_filter is not None
    and test2_parsed_filter.get("regulation_number") == 51
    and test2_parsed_filter.get("regulation_year") == 2023,
)
check(
    "HyDE menghasilkan 2 string berbeda",
    len(_demo_hyde_answers) == 2 and _demo_hyde_answers[0] != _demo_hyde_answers[1],
)
check("Jumlah kandidat sebelum rerank >= 5 (demo)", len(_demo_candidates_preview) >= 5)
check("Tabel rerank memiliki TOP_K_FINAL baris (demo)", len(_rerank_demo) == TOP_K_FINAL)
check(
    "GRPO_MODEL_ID sesuai pola '<username>/pgabl-legal-grpo-smollm2-135m-galih'",
    GRPO_MODEL_ID.endswith("/pgabl-legal-grpo-smollm2-135m-galih"),
)
check(
    "Uji wajib #1 (uang lembur) berjalan tanpa exception",
    isinstance(test1_result, str) and not test1_result.startswith("**Terjadi kesalahan"),
)

n_pass = sum(1 for _, s in checklist_results if s == "PASS")
n_total = len(checklist_results)
print(f"\nRingkasan: {n_pass}/{n_total} item lolos verifikasi.")


[PASS] Jumlah parent documents > 0
[PASS] Jumlah child documents > 0
[PASS] Dimensi embedding > 0
[PASS] Parsing filter PP 51/2023 sesuai (regulation_number=51, regulation_year=2023)
[PASS] HyDE menghasilkan 2 string berbeda
[PASS] Jumlah kandidat sebelum rerank >= 5 (demo)
[PASS] Tabel rerank memiliki TOP_K_FINAL baris (demo)
[PASS] GRPO_MODEL_ID sesuai pola '<username>/pgabl-legal-grpo-smollm2-135m-galih'
[PASS] Uji wajib #1 (uang lembur) berjalan tanpa exception

Ringkasan: 9/9 item lolos verifikasi.
